# 02 — Event Model & Possession Reconstruction

This notebook transforms the raw event table from **Notebook 01** into analysis-ready datasets for downstream modelling.

## Objectives

1. Load the `events_raw` table created in Notebook 01.
2. Optionally enrich the event table with a minimal set of outcome columns required for modelling.
3. Reconstruct possession-level summaries.
4. Build an **action-level dataset** for on-ball value modelling.
5. Create early labels and state features that will feed the **xT** and **VAEP-style** notebooks.

## Outputs

- `data/processed/events_enriched.parquet`
- `data/processed/possessions.parquet`
- `data/processed/actions.parquet`
- `db/football_value.duckdb`
  - `events_enriched`
  - `possessions`
  - `actions`


## 0. Setup

In [ ]:
from __future__ import annotations

import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)
warnings.filterwarnings("ignore")


In [ ]:
try:
    import duckdb
except Exception as e:
    raise ImportError("duckdb is required. Install it with: pip install duckdb") from e

try:
    from tqdm.auto import tqdm
except Exception as e:
    raise ImportError("tqdm is required. Install it with: pip install tqdm") from e

try:
    from statsbombpy import sb
    STATSBOMBPY_AVAILABLE = True
except Exception:
    STATSBOMBPY_AVAILABLE = False


## 1. Project paths

In [ ]:
def find_repo_root(start: Optional[Path] = None) -> Path:
    start = start or Path.cwd()
    for p in [start, *start.parents]:
        if (p / "README.md").exists():
            return p
    return start

REPO_ROOT = find_repo_root()
DATA_DIR = REPO_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
DB_DIR = REPO_ROOT / "db"

for d in [RAW_DIR, PROCESSED_DIR, DB_DIR]:
    d.mkdir(parents=True, exist_ok=True)

DB_PATH = DB_DIR / "football_value.duckdb"
BASE_EVENTS_PATH = PROCESSED_DIR / "events.parquet"
MATCHES_PATH = PROCESSED_DIR / "matches.csv"

ENRICHED_EVENTS_PATH = PROCESSED_DIR / "events_enriched.parquet"
POSSESSIONS_PATH = PROCESSED_DIR / "possessions.parquet"
ACTIONS_PATH = PROCESSED_DIR / "actions.parquet"

REPO_ROOT, BASE_EVENTS_PATH, DB_PATH


## 2. Configuration

In [ ]:
@dataclass(frozen=True)
class Config:
    force_event_enrichment: bool = False
    save_outputs: bool = True
    vaep_horizon_actions: int = 10
    progressive_threshold: float = 5.0
    pitch_length: float = 120.0
    pitch_width: float = 80.0

CFG = Config()
CFG


## 3. Load raw events from Notebook 01

Expected input:
- `data/processed/events.parquet`
- `db/football_value.duckdb` table `events_raw`

If the minimal modelling fields are missing, the notebook can optionally re-fetch a compact enriched view from StatsBomb Open Data.


In [ ]:
if not BASE_EVENTS_PATH.exists():
    raise FileNotFoundError(
        f"Base events file not found at {BASE_EVENTS_PATH}. Run Notebook 01 first."
    )

events_raw = pd.read_parquet(BASE_EVENTS_PATH)
print(events_raw.shape)
events_raw.head()


In [ ]:
required_for_modelling = [
    "match_id", "event_id", "index", "period", "minute", "second",
    "possession", "possession_team", "team", "player", "event_type",
    "play_pattern", "under_pressure", "x_start", "y_start", "x_end", "y_end"
]

missing_required = [c for c in required_for_modelling if c not in events_raw.columns]
if missing_required:
    raise ValueError(f"Notebook 01 output is missing required columns: {missing_required}")

events_raw["event_type"].value_counts().head(15)


## 4. Minimal enrichment step (optional but recommended)

The raw table created in Notebook 01 is intentionally compact.  
For action valuation, a few event-specific outcome fields are useful:

- `shot_outcome`
- `pass_outcome`
- `dribble_outcome`
- `pass_type`

If `events_enriched.parquet` already exists, it will be loaded directly.  
Otherwise, the notebook can rebuild a compact enriched table from StatsBomb Open Data.


In [ ]:
def safe_get_xy(v):
    if isinstance(v, (list, tuple)) and len(v) >= 2:
        return float(v[0]), float(v[1])
    return np.nan, np.nan

def needs_enrichment(df: pd.DataFrame) -> bool:
    desired = {"shot_outcome", "pass_outcome", "dribble_outcome", "pass_type"}
    return len(desired.intersection(df.columns)) < len(desired)

need_enrichment = CFG.force_event_enrichment or needs_enrichment(events_raw)
print("Need enrichment:", need_enrichment)
print("statsbombpy available:", STATSBOMBPY_AVAILABLE)


In [ ]:
if ENRICHED_EVENTS_PATH.exists() and not CFG.force_event_enrichment:
    events = pd.read_parquet(ENRICHED_EVENTS_PATH)
    print(f"Loaded existing enriched events: {events.shape}")
else:
    if need_enrichment:
        if not STATSBOMBPY_AVAILABLE:
            raise ImportError(
                "statsbombpy is required for event enrichment. Install it or disable enrichment."
            )
        if not MATCHES_PATH.exists():
            raise FileNotFoundError(
                f"Matches file not found at {MATCHES_PATH}. Run Notebook 01 first."
            )

        matches = pd.read_csv(MATCHES_PATH)
        match_ids = matches["match_id"].astype(int).tolist()

        enriched_parts = []
        for mid in tqdm(match_ids, desc="Enriching events from StatsBomb"):
            ev = sb.events(match_id=mid)
            ev["match_id"] = mid

            ev["x_start"], ev["y_start"] = zip(*ev["location"].map(safe_get_xy))

            if "pass_end_location" in ev.columns:
                px, py = zip(*ev["pass_end_location"].map(safe_get_xy))
                ev["x_end"], ev["y_end"] = px, py
            else:
                ev["x_end"], ev["y_end"] = np.nan, np.nan

            if "carry_end_location" in ev.columns:
                cx, cy = zip(*ev["carry_end_location"].map(safe_get_xy))
                ev["x_end"] = ev["x_end"].where(~ev["x_end"].isna(), cx)
                ev["y_end"] = ev["y_end"].where(~ev["y_end"].isna(), cy)

            if "shot_end_location" in ev.columns:
                sx, sy = zip(*ev["shot_end_location"].map(safe_get_xy))
                ev["x_end"] = ev["x_end"].where(~ev["x_end"].isna(), sx)
                ev["y_end"] = ev["y_end"].where(~ev["y_end"].isna(), sy)

            keep_cols = [
                "match_id", "id", "index", "period", "minute", "second",
                "possession", "possession_team", "team", "player", "type",
                "play_pattern", "under_pressure", "x_start", "y_start", "x_end", "y_end",
                "shot_outcome", "pass_outcome", "dribble_outcome", "pass_type",
            ]
            keep_cols = [c for c in keep_cols if c in ev.columns]
            ev = ev[keep_cols].copy()
            ev = ev.rename(columns={"id": "event_id", "type": "event_type"})
            enriched_parts.append(ev)

        events = pd.concat(enriched_parts, ignore_index=True)
        if CFG.save_outputs:
            events.to_parquet(ENRICHED_EVENTS_PATH, index=False)
            print(f"Saved enriched events to: {ENRICHED_EVENTS_PATH}")
    else:
        events = events_raw.copy()
        print("Using base events table without enrichment.")

print(events.shape)
events.head()


## 5. Standardise event model

In [ ]:
events = events.copy()
events = events.sort_values(["match_id", "index"], kind="mergesort").reset_index(drop=True)

events["match_seconds"] = (
    (events["period"].fillna(1).astype(int) - 1) * 45 * 60
    + events["minute"].fillna(0).astype(int) * 60
    + events["second"].fillna(0).astype(int)
)

events["possession_uid"] = (
    events["match_id"].astype(str) + "_" + events["possession"].astype(str)
)

events["is_pass"] = events["event_type"].eq("Pass")
events["is_carry"] = events["event_type"].eq("Carry")
events["is_shot"] = events["event_type"].eq("Shot")
events["is_dribble"] = events["event_type"].eq("Dribble")
events["is_receipt"] = events["event_type"].eq("Ball Receipt*")

if "shot_outcome" in events.columns:
    events["is_goal"] = events["shot_outcome"].astype(str).eq("Goal")
else:
    events["is_goal"] = False

if "pass_outcome" in events.columns:
    events["pass_completed"] = np.where(events["is_pass"], events["pass_outcome"].isna(), np.nan)
else:
    events["pass_completed"] = np.nan

if "dribble_outcome" in events.columns:
    events["dribble_completed"] = np.where(
        events["is_dribble"],
        events["dribble_outcome"].astype(str).eq("Complete"),
        np.nan
    )
else:
    events["dribble_completed"] = np.nan

events["action_success"] = np.select(
    [
        events["is_pass"] & events["pass_completed"].fillna(False),
        events["is_carry"],
        events["is_dribble"] & events["dribble_completed"].fillna(False),
        events["is_shot"] & events["is_goal"],
    ],
    [1, 1, 1, 1],
    default=0
)

goal_x = CFG.pitch_length
goal_y = CFG.pitch_width / 2

events["dist_to_goal_start"] = np.sqrt((goal_x - events["x_start"])**2 + (goal_y - events["y_start"])**2)
events["dist_to_goal_end"] = np.sqrt((goal_x - events["x_end"])**2 + (goal_y - events["y_end"])**2)
events["delta_goal_distance"] = events["dist_to_goal_start"] - events["dist_to_goal_end"]

events["delta_x"] = events["x_end"] - events["x_start"]
events["delta_y"] = events["y_end"] - events["y_start"]
events["move_length"] = np.sqrt(events["delta_x"]**2 + events["delta_y"]**2)

events["is_progressive_action"] = events["delta_goal_distance"] >= CFG.progressive_threshold

events.head()


## 6. Possession reconstruction

In [ ]:
events["event_number_in_possession"] = (
    events.groupby("possession_uid").cumcount() + 1
)

possessions = (
    events.groupby(["match_id", "possession", "possession_uid", "possession_team"], dropna=False)
    .agg(
        start_index=("index", "min"),
        end_index=("index", "max"),
        start_period=("period", "min"),
        end_period=("period", "max"),
        start_minute=("minute", "min"),
        end_minute=("minute", "max"),
        start_second=("second", "min"),
        end_second=("second", "max"),
        start_match_seconds=("match_seconds", "min"),
        end_match_seconds=("match_seconds", "max"),
        n_events=("event_id", "count"),
        n_players=("player", "nunique"),
        start_x=("x_start", "first"),
        start_y=("y_start", "first"),
        end_x=("x_end", "last"),
        end_y=("y_end", "last"),
        n_shots=("is_shot", "sum"),
        n_goals=("is_goal", "sum"),
        n_progressive_actions=("is_progressive_action", "sum"),
    )
    .reset_index()
)

possessions["duration_seconds"] = (
    possessions["end_match_seconds"] - possessions["start_match_seconds"]
).clip(lower=0)

possessions["ends_in_shot"] = possessions["n_shots"] > 0
possessions["ends_in_goal"] = possessions["n_goals"] > 0
possessions["progressive_action_share"] = (
    possessions["n_progressive_actions"] / possessions["n_events"].replace(0, np.nan)
)

possessions.head()


In [ ]:
print("Possessions:", len(possessions))
print("Average events per possession:", round(possessions["n_events"].mean(), 2))
print("Shot-ending possessions:", round(possessions["ends_in_shot"].mean() * 100, 2), "%")
print("Goal-ending possessions:", round(possessions["ends_in_goal"].mean() * 100, 2), "%")


## 7. Build the action universe

For the first modelling version, the action universe is restricted to:

- `Pass`
- `Carry`
- `Dribble`
- `Shot`

This keeps the dataset focused on the on-ball actions that directly change possession state.


In [ ]:
action_types = ["Pass", "Carry", "Dribble", "Shot"]

actions = (
    events.loc[events["event_type"].isin(action_types)].copy()
    .sort_values(["match_id", "index"], kind="mergesort")
    .reset_index(drop=True)
)

actions["action_id"] = np.arange(1, len(actions) + 1)
actions["action_number_in_possession"] = (
    actions.groupby("possession_uid").cumcount() + 1
)
actions["actions_in_possession"] = actions.groupby("possession_uid")["action_id"].transform("count")
actions["remaining_actions_in_possession"] = (
    actions["actions_in_possession"] - actions["action_number_in_possession"]
)

actions["time_since_possession_start"] = (
    actions["match_seconds"]
    - actions.groupby("possession_uid")["match_seconds"].transform("min")
)

actions["is_move_action"] = actions["event_type"].isin(["Pass", "Carry", "Dribble"])
actions["xT_candidate"] = np.select(
    [
        actions["is_pass"] & actions["pass_completed"].fillna(False),
        actions["is_carry"],
        actions["is_dribble"] & actions["dribble_completed"].fillna(False),
    ],
    [1, 1, 1],
    default=0
)

actions[[
    "match_id", "event_id", "event_type", "team", "player",
    "x_start", "y_start", "x_end", "y_end",
    "delta_goal_distance", "is_progressive_action", "xT_candidate"
]].head(10)


## 8. Create forward-looking labels for VAEP-style modelling

We create a first-pass approximation of action-value labels over the next **K actions** in the same match.

Labels:
- `goal_for_next_k_actions`
- `goal_against_next_k_actions`

This is a practical, portfolio-friendly baseline.  
It captures forward scoring consequences without introducing future-state leakage into the feature set itself.


In [ ]:
K = CFG.vaep_horizon_actions

actions["goal_for_next_k_actions"] = 0
actions["goal_against_next_k_actions"] = 0

for match_id, idx in tqdm(actions.groupby("match_id").groups.items(), desc="Labelling future goals"):
    idx = np.array(list(idx))
    sub = actions.loc[idx].copy()

    teams = sub["team"].astype(str).values
    goals = sub["is_goal"].fillna(False).astype(bool).values
    n = len(sub)

    gf = np.zeros(n, dtype=np.int8)
    ga = np.zeros(n, dtype=np.int8)

    for i in range(n):
        j = min(i + K + 1, n)
        if i + 1 >= j:
            continue

        future_goals = goals[i + 1:j]
        future_teams = teams[i + 1:j]

        if future_goals.any():
            gf[i] = int(np.any(future_goals & (future_teams == teams[i])))
            ga[i] = int(np.any(future_goals & (future_teams != teams[i])))

    actions.loc[idx, "goal_for_next_k_actions"] = gf
    actions.loc[idx, "goal_against_next_k_actions"] = ga

actions[[
    "match_id", "index", "team", "event_type", "is_goal",
    "goal_for_next_k_actions", "goal_against_next_k_actions"
]].head(15)


In [ ]:
possession_goal_map = possessions.set_index("possession_uid")["ends_in_goal"]
possession_shot_map = possessions.set_index("possession_uid")["ends_in_shot"]

actions["goal_in_current_possession"] = actions["possession_uid"].map(possession_goal_map).fillna(False).astype(int)
actions["shot_in_current_possession"] = actions["possession_uid"].map(possession_shot_map).fillna(False).astype(int)

actions[[
    "event_type", "goal_for_next_k_actions", "goal_against_next_k_actions",
    "goal_in_current_possession", "shot_in_current_possession"
]].head()


## 9. Quality checks

In [ ]:
print("Total actions:", len(actions))
print("Unique matches:", actions["match_id"].nunique())
print("Action types:")
print(actions["event_type"].value_counts())
print("\nGoal labels:")
print(actions[["goal_for_next_k_actions", "goal_against_next_k_actions", "goal_in_current_possession"]].mean().round(4))


In [ ]:
qa = pd.DataFrame({
    "metric": [
        "raw_events",
        "possessions",
        "actions",
        "unique_matches_actions",
        "xT_candidates",
        "progressive_actions",
        "goals",
    ],
    "value": [
        len(events),
        len(possessions),
        len(actions),
        actions["match_id"].nunique(),
        int(actions["xT_candidate"].sum()),
        int(actions["is_progressive_action"].sum()),
        int(actions["is_goal"].sum()),
    ],
})
qa


## 10. Save outputs

In [ ]:
if CFG.save_outputs:
    events.to_parquet(ENRICHED_EVENTS_PATH, index=False)
    possessions.to_parquet(POSSESSIONS_PATH, index=False)
    actions.to_parquet(ACTIONS_PATH, index=False)

    try:
        con.close()
    except:
        pass

    con = duckdb.connect(str(DB_PATH))
    con.execute(
        "CREATE OR REPLACE TABLE events_enriched AS SELECT * FROM read_parquet(?)",
        [str(ENRICHED_EVENTS_PATH)]
    )
    con.execute(
        "CREATE OR REPLACE TABLE possessions AS SELECT * FROM read_parquet(?)",
        [str(POSSESSIONS_PATH)]
    )
    con.execute(
        "CREATE OR REPLACE TABLE actions AS SELECT * FROM read_parquet(?)",
        [str(ACTIONS_PATH)]
    )
    con.close()

    print(f"Saved enriched events to: {ENRICHED_EVENTS_PATH}")
    print(f"Saved possessions to:    {POSSESSIONS_PATH}")
    print(f"Saved actions to:        {ACTIONS_PATH}")
    print(f"Updated DuckDB at:       {DB_PATH}")

## 11. Preview from DuckDB

In [ ]:
con = duckdb.connect(str(DB_PATH))

summary_sql = '''
SELECT
    COUNT(*) AS n_actions,
    COUNT(DISTINCT match_id) AS n_matches,
    SUM(is_goal) AS n_goals,
    AVG(goal_for_next_k_actions) AS goal_for_rate,
    AVG(goal_against_next_k_actions) AS goal_against_rate
FROM actions
'''
con.execute(summary_sql).df()


# Conclusions

## Event Modelling Output

The event modelling stage successfully transformed the raw StatsBomb event logs into a structured analytical dataset suitable for downstream football analytics modelling.

After processing **88 matches**, the pipeline produced:

* **307,721 raw events**
* **16,255 reconstructed possessions**
* **158,114 on-ball actions**

These outputs represent the transition from raw event logs to a modelling-ready football dataset.

Three persistent tables were generated:

* `events_enriched` → full event-level dataset with spatial and contextual attributes
* `possessions` → possession-level summaries and outcomes
* `actions` → filtered on-ball action universe for possession value modelling

These datasets are stored both as **Parquet files and DuckDB tables**, enabling efficient analytical queries and reproducible pipelines.

---

## Possession Structure

Possession reconstruction produced **16,255 possessions** across the dataset.

Key descriptive statistics:

* **Average events per possession:** ~18.9
* **Shot-ending possessions:** ~13.0%
* **Goal-ending possessions:** ~1.56%

These values are consistent with known empirical distributions in professional football event data.

The possession layer therefore provides a reliable structure for analysing:

* attacking sequences
* progression efficiency
* shot creation dynamics

---

## Action Universe

From the full event stream, the modelling universe was restricted to four primary on-ball actions:

* **Pass**
* **Carry**
* **Dribble**
* **Shot**

The resulting distribution:

| Action  | Count  |
| ------- | ------ |
| Pass    | 84,626 |
| Carry   | 67,907 |
| Dribble | 3,284  |
| Shot    | 2,297  |

This design focuses the modelling dataset on actions that **directly alter the attacking state of possession**, which is critical for possession value models such as **Expected Threat (xT)** and **VAEP**.

---

## Spatial Progression Signals

The feature engineering stage introduced spatial progression metrics derived from the StatsBomb pitch coordinate system (120x80).

Key variables include:

* distance to goal
* change in distance to goal
* movement length
* progressive action indicator

Across the dataset:

* **57,100 progressive actions** were identified
* **136,522 actions qualify as xT candidates**

These features form the spatial backbone for modelling **ball progression value across the pitch**.

---

## Forward Outcome Labels

To support supervised modelling, forward-looking labels were created over the next **K = 10 actions** within each match.

Observed label rates:

| Label                       | Rate  |
| --------------------------- | ----- |
| goal_for_next_k_actions     | 1.22% |
| goal_against_next_k_actions | 0.30% |
| goal_in_current_possession  | 1.45% |

These low base rates reflect the **inherent sparsity of goals in football**, which is an important modelling constraint for action valuation frameworks.

---

## Data Engineering Result

At the end of this notebook, the project has produced a **fully structured event modelling layer**, including:

* action-level state features
* possession segmentation
* spatial progression signals
* forward outcome labels

This modelling layer enables multiple downstream analyses, including:

* Expected Threat (xT)
* VAEP-style action valuation
* player contribution analysis
* tactical sequence modelling

---

## Project Significance

This notebook represents the **transition point from data engineering to football analytics modelling**.

The dataset produced here forms the foundation for the next stages of the project:

* **Notebook 03 — Expected Threat (xT) modelling**
* spatial value estimation across the pitch
* player-level ball progression contribution analysis
